[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_1_Colab.ipynb)

# LDSC Practical 1 — SNP Heritability (Colab version)

**MadhurBain Singh, Benjamin Neale**  
**March 5th, 2025**

This notebook is a Colab-friendly rewrite of the original R practical.  
It keeps the same analysis flow, but replaces local paths with a Colab setup and adds a short installation section.

**Before you start**
- Open this notebook in Colab.
- Run the setup cells top to bottom.
- Make sure the `EUR/` folder and the input files used in the practical are available in your Colab session (for example by uploading them or mounting Google Drive).


## Qualtrics Link

https://qimr.az1.qualtrics.com/jfe/form/SV_29tZDm8QmlN31Q2


## 1) Environment setup / required packages

This Colab notebook uses an **R runtime**. We install the packages needed for the practical from CRAN and GitHub.

Packages used here:
- `data.table`
- `dplyr`
- `remotes`
- `GenomicSEM`

If Colab asks you to restart the runtime after installation, do that once and then continue from the import cell.


In [ ]:
# System dependencies that are often useful for R package installation in Colab
# (safe to run even if some packages are already present)
system("apt-get update -qq")
system("apt-get install -y -qq libcurl4-openssl-dev libxml2-dev libssl-dev")
system('python -m pip -q install gdown')
system('gdown --folder "https://drive.google.com/drive/folders/1BIBaoLlodLG_ni9DASh_bw-3RzUZSSJe?usp=sharing" -O /content/LDSC_Practical_1')
root_dir <- "/content/LDSC_Practical_1"
sumstat_dir <- file.path(root_dir, "EUR")
ref_data_dir <- file.path(root_dir, "EUR", "eur_w_ld_chr")

In [ ]:
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes"),
  repos = "https://cloud.r-project.org"
)

# Install GenomicSEM from GitHub
# GenomicSEM is maintained on GitHub and its README recommends installation from GitHub.
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never")


### ✅ Load packages

If the install cell ran successfully, load the packages here.


In [ ]:
library(data.table)
library(dplyr)
library(GenomicSEM)

sessionInfo()


## 2) Set up the working directory in Colab

Unlike the original local practical, Colab does not have your course folder on disk.

Choose one of these options:
- upload the practical files directly into the Colab session, or
- mount Google Drive and point `root_dir` to the folder containing the practical.

The code below assumes you have a folder with this structure:

```text
root_dir/
  EUR/
    w_hm3.snplist
    eur_w_ld_chr/
    CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
    BMI.sumstats.gz
```


In [ ]:
# Edit this path to match where you placed the practical files in Colab.
# A common choice is a folder in Google Drive, or /content if you upload files directly.
root_dir <- "/content/LDSC_Practical_1"

if (!dir.exists(root_dir)) {
  dir.create(root_dir, recursive = TRUE, showWarnings = FALSE)
}

setwd(root_dir)

# Set directories
sumstat_dir  <- "EUR/"
ref_data_dir <- "EUR/eur_w_ld_chr/"

# Quick check
getwd()


## 3) HapMap3 SNPs and LD scores in European ancestry

We first inspect the HapMap3 SNP list used by LDSC.


### HapMap3 SNPs

**Question:** How many SNPs are there in the HapMap3 reference file?


In [ ]:
hm3_file <- file.path(ref_data_dir, "w_hm3.snplist")

# Take a look at the first few lines
system(paste0("head ", hm3_file))

# Count the number of lines
system(paste0("wc -l ", hm3_file))


## 4) QC filters for LDSC


In [ ]:
info_cutoff <- 0.9
maf_cutoff  <- 0.01


## 5) BMI example: subset to chromosome 22

For the tutorial, we practice `munge()` using only chromosome 22.


In [ ]:
bmi_file <- file.path(
  sumstat_dir,
  "CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz"
)

system(paste0("zcat ", bmi_file, " | head"))


### Rename columns as needed

Renaming is not essential if LDSC can already interpret the column names, but it is useful when the summary statistics use non-standard names.


In [ ]:
bmi_GWAS <- fread(bmi_file, header = TRUE, data.table = FALSE)
str(bmi_GWAS)

bmi_GWAS <- dplyr::rename(
  bmi_GWAS,
  A1 = Tested_Allele,
  A2 = Other_Allele
)

head(bmi_GWAS)


In [ ]:
bmi_updated_file <- file.path(sumstat_dir, "BMI_GWAS_for_LDSC.txt")

fwrite(
  bmi_GWAS,
  file = bmi_updated_file,
  sep = "\t",
  col.names = TRUE,
  row.names = TRUE,
  quote = FALSE
)

system(paste0("head ", bmi_updated_file))


### Munge

Provide the file path and name to save the munged summary statistics.


In [ ]:
bmi_munged <- file.path(sumstat_dir, "munged_BMI_GWAS_chr22_for_LDSC")

munge(
  files = bmi_updated_file,
  hm3 = hm3_file,
  trait.names = bmi_munged,
  info.filter = info_cutoff,
  maf.filter = maf_cutoff
)


## 6) LDSC

The full BMI summary statistics are already munged for the practical. We compare the raw file with the munged file and then run LDSC.


In [ ]:
raw_bmi    <- file.path(sumstat_dir, "Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz")
munged_bmi <- file.path(sumstat_dir, "Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz")

system(paste0("zcat ", raw_bmi, " | head"))
system(paste0("zcat ", munged_bmi, " | head"))

system(paste0("zcat ", raw_bmi, " | wc -l"))
system(paste0("zcat ", munged_bmi, " | wc -l"))


In [ ]:
bmi_ldsc_out <- file.path(sumstat_dir, "BMI")

ldsc(
  traits = munged_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_ldsc_out
)


**Note:** the `ldsc()` function in `GenomicSEM` is designed for bivariate LDSC, so it may print a warning about needing two or more traits. In this practical, that warning can be ignored.


## 7) Bonus: BMI GWAS with LMM

This section uses the already munged BMI summary statistics from Pan-UKBB with a linear mixed model (SAIGE).


In [ ]:
munged_lmm_bmi <- file.path(sumstat_dir, "BMI.sumstats.gz")

system(paste0("zcat ", munged_lmm_bmi, " | head"))


In [ ]:
bmi_lmm_ldsc_out <- file.path(sumstat_dir, "BMI_LMM")

ldsc(
  traits = munged_lmm_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI_lmm",
  ldsc.log = bmi_lmm_ldsc_out
)


## End

You have now run the full Colab version of Practical 1 for SNP heritability.
